<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# C18's Two Loose Ends

In [C18's run](c18_cliff_run.ipynb) the conserved budget at c = 0.6 (a gain of 1.2) located its cliff at D = 1.528, not
near d = 1, the graph's own percolation point. The candidate written down there, *a hunch*: the activation threshold of
0.1 raises the effective percolation point above the graph's. This notebook sweeps that threshold at the same gain, as
registered in the journal of 2026-11-03, § C18's Loose Ends: the Plan. The model, the graph and the dynamics are the
run's, copied unchanged; only the threshold that counts a unit as reached is varied, and all thresholds are read off
the same settled activations of the same draws. The graph's own reach (the seed's directed out-component) is read off
the same draws too.

The second loose end is the 1.1 D reading. In the run, D was interpolated on a grid of step 0.25, so it could sit a
grid step before a jump, and 1.1 D need not clear it: the jump control, whose jump is at 2.5, read a share of 0.000.
Here D is located again on a grid of step 0.02 near the pure network's threshold, for the pure network and the jump
control, and the 1.1 D share is read again at that D.


In [1]:
import multiprocessing as mp
import os
import warnings

import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*fork.*")

SEED, STAGE = 20261103, 11  # the run used stages 0 to 4; stage 11 is new
WORKERS = 2
N, DRAWS, BUDGET = 10000, 50, 0.6
GRID = np.arange(0.5, 3.0001, 0.125)
THRESHOLDS = [0.1, 0.03, 0.01, 0.003, 0.001]
REACH_LEVEL = 0.01
SETTLE_TOL, SETTLE_STEPS = 1e-10, 20000
EDGE, DECAY, INHIBITION = 0.2, 0.5, 0.5
ACTIVE, JUMP_AT = 0.1, 2.5
FINE = np.arange(2.0, 3.0001, 0.02)
SHARE_DRAWS = 200


def network(n, d, rng):
    """The pilot's directed random graph: Poisson(d n) edges with uniform ends, self-loops dropped, repeats kept."""
    m = rng.poisson(d * n)
    src, dst = rng.integers(0, n, m), rng.integers(0, n, m)
    keep = src != dst
    return src[keep], dst[keep]


def out_component(n, src, dst, seed=0):
    """A mask of the units the seed reaches along directed edges, the seed included."""
    reached = np.zeros(n, dtype=bool)
    reached[seed] = True
    frontier = np.array([seed])
    while frontier.size:
        on = np.zeros(n, dtype=bool)
        on[frontier] = True
        new = dst[on[src]]
        new = np.unique(new[~reached[new]])
        reached[new] = True
        frontier = new
    return reached


def steady(n, src, dst, budget=None, fanout=False, inhibition=False):
    """The plan's dynamics: the seed held at 1; each step a unit becomes half its last value plus its weighted input,
    clipped to [0, 1]; until no unit moves by more than 1e-10 or for 20 000 steps. Returns the activations, whether
    they settled, and the steps taken."""
    outdeg = np.bincount(src, minlength=n)
    indeg = np.bincount(dst, minlength=n)
    if budget is not None:  # each unit's output, budget x activation, divided equally among its targets
        w = budget / outdeg[src]
    elif fanout:  # 0.2 / (1 + ln k) for a source of out-degree k
        w = EDGE / (1.0 + np.log(outdeg[src]))
    else:
        w = np.full(src.size, EDGE)
    a = np.zeros(n)
    a[0] = 1.0
    for step in range(1, SETTLE_STEPS + 1):
        drive = np.bincount(dst, weights=w * a[src], minlength=n)
        if inhibition:  # minus half the mean activation of a unit's inputs
            drive = drive - INHIBITION * np.bincount(dst, weights=a[src], minlength=n) / np.maximum(indeg, 1)
        new = np.clip(DECAY * a + drive, 0.0, 1.0)
        new[0] = 1.0
        moved = np.max(np.abs(new - a))
        a = new
        if moved <= SETTLE_TOL:
            return a, True, step
    return a, False, SETTLE_STEPS


def one_draw(point, i):
    """One eligible draw at GRID[point]: the fraction of the other units above each threshold, the out-component's
    fraction, whether it settled, and how many graphs were replaced for a seed with no out-edge."""
    rng = np.random.default_rng([SEED, STAGE, point, i])
    replaced = 0
    while True:
        src, dst = network(N, float(GRID[point]), rng)
        if np.any(src == 0):
            break
        replaced += 1
    a, settled, _ = steady(N, src, dst, budget=BUDGET)
    above = [float(np.mean(a[1:] > t)) for t in THRESHOLDS]
    topo = float((out_component(N, src, dst).sum() - 1) / (N - 1))
    return above, topo, settled, replaced


def column(point):
    return [one_draw(point, i) for i in range(DRAWS)]


def locate(means):
    """The run's rule: D is where the mean reached fraction first passes 1%, interpolated linearly from the grid
    point before; None if it never does."""
    passing = means >= REACH_LEVEL
    if not passing.any():
        return None
    i = int(np.argmax(passing))
    if i == 0:
        return float(GRID[0])
    return float(GRID[i - 1] + (REACH_LEVEL - means[i - 1]) * (GRID[i] - GRID[i - 1]) / (means[i] - means[i - 1]))


def locate_on(grid, means):
    passing = means >= REACH_LEVEL
    if not passing.any():
        return None
    i = int(np.argmax(passing))
    if i == 0:
        return float(grid[0])
    return float(grid[i - 1] + (REACH_LEVEL - means[i - 1]) * (grid[i] - grid[i - 1]) / (means[i] - means[i - 1]))


def pure_or_jump(task):
    """One draw of the pure network (threshold 0.1) or of the run's jump control at mean degree d."""
    kind, stage, point, i, d = task
    rng = np.random.default_rng([SEED, stage, ("pure", "jump").index(kind), point, i])
    while True:
        src, dst = network(N, d, rng)
        if np.any(src == 0):
            break
    if kind == "jump":
        if d < JUMP_AT:
            return np.unique(dst[src == 0]).size / (N - 1), True
        return (out_component(N, src, dst).sum() - 1) / (N - 1), True
    a, settled, _ = steady(N, src, dst)
    return float(np.mean(a[1:] > ACTIVE)), settled


# Harness check: at threshold 0.1 the copied dynamics give the run's harness answers on a complete graph of 100.
complete = np.array([(i, j) for i in range(100) for j in range(100) if i != j]).T
a1, s1, _ = steady(100, complete[0], complete[1])
assert s1 and np.all(a1[1:] > 0.1)
print("numpy", np.__version__, f"- generator seeds [{SEED}, {STAGE}, point, draw]; {WORKERS} worker processes")


numpy 2.5.4 - generator seeds [20261103, 11, point, draw]; 2 worker processes


## The sweep

Every draw at every grid point, read at every threshold.


In [2]:
with mp.get_context("fork").Pool(WORKERS) as pool:
    cols = pool.map(column, range(GRID.size), chunksize=1)
above = np.array([[r[0] for r in col] for col in cols])  # (grid, draw, threshold)
topo = np.array([[r[1] for r in col] for col in cols])
unsettled = sum(not r[2] for col in cols for r in col)
replaced = sum(r[3] for col in cols for r in col)
means = above.mean(axis=1)
D = {t: locate(means[:, j]) for j, t in enumerate(THRESHOLDS)}
D_topo = locate(topo.mean(axis=1))
print(f"{GRID.size} grid points x {DRAWS} draws at N = {N}; unsettled {unsettled}; graphs replaced {replaced}")
print("threshold   D")
for t in THRESHOLDS:
    print(f"{t:<9}   {'none' if D[t] is None else f'{D[t]:.3f}'}")
print(f"out-component   {D_topo:.3f}")
print("\nmean reached fraction by d (columns: thresholds, then the out-component)")
for k, d in enumerate(GRID):
    print(f"{d:5.3f}  " + "  ".join(f"{m:.4f}" for m in means[k]) + f"  {topo[k].mean():.4f}")


21 grid points x 50 draws at N = 10000; unsettled 0; graphs replaced 363
threshold   D
0.1         1.654
0.03        1.522
0.01        1.377
0.003       1.216
0.001       1.160
out-component   1.014

mean reached fraction by d (columns: thresholds, then the out-component)
0.500  0.0002  0.0002  0.0002  0.0002  0.0002  0.0002
0.625  0.0004  0.0004  0.0004  0.0004  0.0004  0.0004
0.750  0.0005  0.0005  0.0005  0.0005  0.0005  0.0005
0.875  0.0007  0.0009  0.0010  0.0011  0.0011  0.0012
1.000  0.0011  0.0016  0.0020  0.0026  0.0030  0.0045
1.125  0.0011  0.0019  0.0030  0.0044  0.0062  0.0527
1.250  0.0016  0.0035  0.0065  0.0121  0.0198  0.1735
1.375  0.0019  0.0046  0.0098  0.0236  0.0489  0.3327
1.500  0.0022  0.0073  0.0227  0.0696  0.1457  0.4309
1.625  0.0039  0.0225  0.0882  0.2317  0.3752  0.5899
1.750  0.0304  0.1125  0.2494  0.4140  0.5186  0.6147
1.875  0.3857  0.5647  0.6380  0.6688  0.6787  0.6845
2.000  0.5268  0.6715  0.7213  0.7398  0.7454  0.7482
2.125  0.5931  0.7096  0.

## The registered reading


In [3]:
# The registered rules (journal of 2026-11-03, § C18's Loose Ends: the Plan).
L0 = D[0.1] is not None and abs(D[0.1] - 1.528) <= 0.125
L1 = all(D[a] is not None and D[b] is not None and D[b] <= D[a] + 0.125 for a, b in zip(THRESHOLDS, THRESHOLDS[1:]))
L2 = D[0.001] is not None and abs(D[0.001] - D_topo) <= 0.25
refuted = D[0.001] is None or D[0.001] >= 1.4
print(f"L0, the run's D reproduced at threshold 0.1 (within 0.125 of 1.528): {L0}")
print(f"L1, D does not rise as the threshold falls (by more than a grid step): {L1}")
print(f"L2, D at 0.001 within 0.25 of the out-component's D: {L2}")
print(f"refutation, D at 0.001 still at or above 1.4: {refuted}")
print("reading:", "void (L0 fails)" if not L0 else "hunch refuted" if refuted or not L1 else
      "hunch supported" if L2 else "partly: D falls but does not reach the graph's own")


L0, the run's D reproduced at threshold 0.1 (within 0.125 of 1.528): False
L1, D does not rise as the threshold falls (by more than a grid step): True
L2, D at 0.001 within 0.25 of the out-component's D: True
refutation, D at 0.001 still at or above 1.4: False
reading: void (L0 fails)


## The second loose end: D on a finer grid


In [4]:
tasks = [(kind, 12, p, i, float(d)) for kind in ("pure", "jump") for p, d in enumerate(FINE) for i in range(DRAWS)]
with mp.get_context("fork").Pool(WORKERS) as pool:
    out = pool.map(pure_or_jump, tasks, chunksize=DRAWS)
vals = np.array([r[0] for r in out]).reshape(2, FINE.size, DRAWS)
fine_unsettled = sum(not r[1] for r in out)
D_fine = {kind: locate_on(FINE, vals[j].mean(axis=1)) for j, kind in enumerate(("pure", "jump"))}
print(f"fine grid {FINE[0]:.2f} to {FINE[-1]:.2f} step 0.02, {DRAWS} draws a point; unsettled {fine_unsettled}")
for kind in ("pure", "jump"):
    print(f"{kind}: D on the fine grid {D_fine[kind]:.3f}")
print("\nmean reached fraction by d (pure, jump)")
for p, d in enumerate(FINE):
    print(f"{d:.2f}  {vals[0, p].mean():.4f}  {vals[1, p].mean():.4f}")


fine grid 2.00 to 3.00 step 0.02, 50 draws a point; unsettled 9
pure: D on the fine grid 2.429
jump: D on the fine grid 2.480

mean reached fraction by d (pure, jump)
2.00  0.0008  0.0002
2.02  0.0008  0.0002
2.04  0.0007  0.0003
2.06  0.0008  0.0002
2.08  0.0008  0.0003
2.10  0.0007  0.0002
2.12  0.0008  0.0003
2.14  0.0008  0.0002
2.16  0.0008  0.0003
2.18  0.0008  0.0002
2.20  0.0007  0.0003
2.22  0.0009  0.0002
2.24  0.0009  0.0002
2.26  0.0008  0.0002
2.28  0.0010  0.0003
2.30  0.0007  0.0003
2.32  0.0007  0.0003
2.34  0.0009  0.0002
2.36  0.0009  0.0003
2.38  0.0009  0.0002
2.40  0.0010  0.0003
2.42  0.0009  0.0003
2.44  0.0217  0.0003
2.46  0.0015  0.0002
2.48  0.1622  0.0003
2.50  0.4308  0.8217
2.52  0.5983  0.8595
2.54  0.7383  0.8436
2.56  0.8027  0.8645
2.58  0.8371  0.9025
2.60  0.8297  0.8499
2.62  0.8548  0.8702
2.64  0.8078  0.9098
2.66  0.8510  0.9121
2.68  0.8560  0.8961
2.70  0.8627  0.8609
2.72  0.8674  0.9188
2.74  0.8885  0.9197
2.76  0.8764  0.9028
2.78  0.8783  

## The 1.1 D share at the finer D


In [5]:
tasks = [(kind, 13, side, i, float(m * D_fine[kind])) for kind in ("pure", "jump") for side, m in enumerate((1.1, 2.0))
         for i in range(SHARE_DRAWS)]
with mp.get_context("fork").Pool(WORKERS) as pool:
    out = pool.map(pure_or_jump, tasks, chunksize=SHARE_DRAWS)
sv = np.array([r[0] for r in out]).reshape(2, 2, SHARE_DRAWS)
share_unsettled = sum(not r[1] for r in out)
share = {kind: sv[j, 0].mean() / sv[j, 1].mean() for j, kind in enumerate(("pure", "jump"))}
print(f"{SHARE_DRAWS} draws at each of 1.1 D and 2 D; unsettled {share_unsettled}")
for j, kind in enumerate(("pure", "jump")):
    print(f"{kind}: 1.1 D = {1.1 * D_fine[kind]:.3f}, mean fraction {sv[j, 0].mean():.4f}; 2 D = {2 * D_fine[kind]:.3f}, "
          f"mean fraction {sv[j, 1].mean():.4f}; share {share[kind]:.3f}")
J0 = share["jump"] >= 0.9
print(f"J0, the jump control now clears its jump (share at least 0.9): {J0}")
print("pure network's reading:", "void (J0 fails)" if not J0 else "a jump" if share["pure"] >= 0.9 else
      "a continuous rise" if share["pure"] <= 0.1 else "neither")


200 draws at each of 1.1 D and 2 D; unsettled 0
pure: 1.1 D = 2.672, mean fraction 0.8493; 2 D = 4.858, mean fraction 0.9919; share 0.856
jump: 1.1 D = 2.728, mean fraction 0.8999; 2 D = 4.960, mean fraction 0.9926; share 0.907
J0, the jump control now clears its jump (share at least 0.9): True
pure network's reading: neither
